# Task 2 — Parte 1: Definição do problema, variáveis, erros e métricas

**Mineração de Dados · ISEP · 2026/27 1.º semestre**

| Secção | O que responde | Tipo de entrega |
|---|---|---|
| 1.1 | Definição formal do problema | relatório |
| 1.2 | Variáveis nas 4 categorias do enunciado | relatório + tabela |
| 1.3 | Falsos positivos vs falsos negativos e critério do limiar | relatório |
| 1.4 | Métricas: principal, complementares, operacionais, estabilidade | relatório |
| 1.5 | Função `avaliar()` | notebook (+ `avaliacao.py`) |

**Convenções deste notebook**

- As decisões do grupo estão marcadas com **[DECISÃO]**. Foram tomadas **antes de qualquer modelo ser treinado** e devem ser confirmadas pela dupla antes da submissão.
- O **conjunto de teste (out–dez/2021) é removido logo na secção 0** e nunca é usado aqui.
- O relatório não deve reproduzir código: usem o texto das células markdown e os números calculados.

### Decisões do grupo (resumo)

| # | Decisão | Escolha | Onde está justificada |
|---|---|---|---|
| 1 | O alerta é sobre a sessão ou sobre o posto? | A previsão é feita **por sessão**; o alerta serve para **vigiar e manter o posto** | 1.1 |
| 2 | Orçamento de alertas **X** | **X = 200 alertas por 1000 sessões** (20 % das sessões) | 1.3 |
| 3 | Categoria do `Order creation time` | **Categoria 4** (excluída por outra razão metodológica) | 1.2 |

## 0. Preparação

**O que fazer:** carregar os dados, derivar o alvo e separar já o teste. Só são lidas as colunas necessárias para esta parte.

In [20]:
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import avaliacao
from avaliacao import avaliar, avaliar_por_mes, limiar_por_orcamento

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

DATA_PATH = Path("Charging_Data_educational.csv")
NORMAL_CAUSES = ["Charging ends normally", "User stops charging"]

# Fronteiras do split cronológico definido na Task 1
TRAIN_END = pd.Timestamp("2021-07-01")   # treino: dez/2019 – jun/2021
VAL_END   = pd.Timestamp("2021-10-01")   # validação: jul–set/2021 (teste: out–dez/2021)

COLUNAS_ORIGINAIS = pd.read_csv(DATA_PATH, nrows=0).columns.str.strip().tolist()

TEMPOS = ["Order creation time", "Start Time", "End Time", "Payment time"]
df = pd.read_csv(DATA_PATH, usecols=["UserID", "end_cause", *TEMPOS], parse_dates=TEMPOS)
df["end_cause"] = df["end_cause"].str.strip()
df["is_Abnormal"] = (~df["end_cause"].isin(NORMAL_CAUSES)).astype(int)

n_total = len(df)
# O teste deixa de existir neste notebook:
dev = df[df["Start Time"] < VAL_END].copy()
train = dev[dev["Start Time"] < TRAIN_END]
val = dev[dev["Start Time"] >= TRAIN_END]
del df

print(f"{len(COLUNAS_ORIGINAIS)} colunas no ficheiro | {n_total:,} sessões no total")
print(f"Desenvolvimento: {len(dev):,} (treino {len(train):,} + validação {len(val):,})")

47 colunas no ficheiro | 441,077 sessões no total
Desenvolvimento: 360,996 (treino 285,227 + validação 75,769)


## 1.1 Definição do problema

**O que fazer**

- [ ] Especificar a unidade de observação, as classes e a derivação do alvo
- [ ] Fixar o momento da previsão
- [ ] Indicar a finalidade e o significado operacional de prever "anómala"
- [ ] Resumir a validação do alvo feita na Task 1 (secção 1.2 do relatório da Task 1)

**Rascunho para o relatório**

| Elemento | Definição |
|---|---|
| Unidade de observação | Uma sessão de carregamento (uma linha = uma transação de um utilizador num posto). |
| Classe positiva (`is_Abnormal = 1`) | Sessão anómala: `end_cause` diferente de "Charging ends normally" e "User stops charging" (13 causas de falha). |
| Classe negativa (`is_Abnormal = 0`) | Sessão normal: fim normal da carga ou paragem pelo próprio utilizador. A paragem voluntária não é uma falha do equipamento nem do veículo. |
| Derivação do alvo | `is_Abnormal = 1` se `end_cause` não pertencer às duas causas normais. A definição foi validada na Task 1: concorda em 100 % das sessões com a flag histórica pré-calculada `user_no_previous_completed_abnormal`, contra 87,96 % se a paragem do utilizador fosse considerada anómala. |
| Momento da previsão | O instante `Start Time` da sessão. Só pode ser usada informação conhecida nesse instante (contexto, meteorologia prevista, tarifa e histórico de sessões já concluídas). |
| Finalidade | Monitorização preventiva, priorização de inspeção e manutenção, e melhoria da qualidade de serviço. |
| Significado de prever "anómala" | Um **alerta de risco elevado** de a sessão terminar em falha. Não é um diagnóstico da causa nem uma garantia de que a falha ocorre; serve para desencadear uma ação (ver 1.3). |

**[DECISÃO]** A previsão é feita ao nível da **sessão** (é a unidade de observação), mas o alerta destina-se sobretudo a apoiar a **vigilância e a manutenção do posto**. As falhas concentram-se em postos que já falharam (relatório da Task 1, secção 5.2), por isso a ação útil é agir sobre o posto e não sobre a sessão individual. Um alerta significa "este posto, neste momento, tem um risco elevado de falha".

In [21]:
# Números de apoio ao 1.1 (apenas treino e validação)
for nome, d in [("treino", train), ("validação", val)]:
    print(f"{nome:10s} n={len(d):7,}  prevalência de anómalas = {d['is_Abnormal'].mean():.1%}")

causas = (dev.groupby("end_cause")["is_Abnormal"]
             .agg(n="size", anomala="max")
             .sort_values("n", ascending=False))
causas["% das sessões"] = (causas["n"] / len(dev) * 100).round(2)
causas

treino     n=285,227  prevalência de anómalas = 18.2%
validação  n= 75,769  prevalência de anómalas = 21.1%


,n,anomala,% das sessões
end_cause,,,
Charging ends normally,165641,0,45.88
User stops charging,127509,0,35.32
Other faults of the charging pile,16917,1,4.69
DC Output Fault,12587,1,3.49
Vehicle communications failure,11620,1,3.22
Faulty charging connection,7014,1,1.94
BMS failure,6584,1,1.82
Failure of non-system control loop components and assemblies,6123,1,1.70
Vehicle charging faults,4768,1,1.32


## 1.2 Variáveis nas 4 categorias do enunciado

**O que fazer**

- [ ] Classificar **todas** as colunas do ficheiro (e as derivadas) numa das 4 categorias
- [ ] Escrever uma justificação curta para cada exclusão
- [ ] Verificar que nenhuma coluna ficou por classificar (a célula abaixo faz essa verificação)

As justificações vêm da Task 1, mas **reescritas**: a de `UserID` não deve citar o conjunto de teste (ver célula de evidência e o texto proposto mais abaixo).

In [22]:
CAT = {1: "1. Preditor elegível",
       2: "2. Só para construir o alvo",
       3: "3. Gerada durante/depois da sessão",
       4: "4. Excluída por outra razão metodológica"}

regras = []   # (variável, categoria, justificação, é derivada?)
def add(nomes, cat, just, derivada=False):
    for n in ([nomes] if isinstance(nomes, str) else nomes):
        regras.append((n, cat, just, derivada))

# --- Categoria 2: só para construir o alvo
add("end_cause", 2, "Define o alvo e só é conhecida no fim da sessão (fuga de informação direta).")

# --- Categoria 3: geradas durante ou depois da sessão
add("End Time", 3, "Só existe quando a sessão termina.")
add("Payment time", 3, "Registado depois do fim da sessão.")
add("Transaction power/kwh", 3, "Energia acumulada durante a sessão; as sessões anómalas têm energia nula ou duração curta (consequência do fim, não causa).")
add(["Electricity cost/Yuan", "Service charge/Yuan", "Transaction Amount/Yuan", "Actual Payment/Yuan"],
    3, "Calculados a partir da energia consumida; só são conhecidos no fim da sessão.")

# --- Categoria 4: outra razão metodológica
add("Order creation time", 4, "Na maioria das sessões coincide com o início, mas em parte das sessões é registada perto do fim (fuga de informação) e o sentido da diferença está associado ao alvo; não é fiável como informação de início.")
add("UserID", 4, "Identificador de cardinalidade muito alta; grande parte das sessões de validação é de contas não vistas no treino (não generaliza) e identifica pessoas (privacidade). O comportamento do utilizador entra pelos atributos user_*.")
add("tariff_period", 4, "Função determinística de `hour` (redundante).")
add("is_new_user", 4, "Redundante com user_no_previous_completed_session (ρ ≈ 0,99) e inflacionado no arranque dos dados.")
add(["user_previous_sessions_730D", "user_previous_abnormal_730D", "user_previous_abnormal_rate_730D"],
    4, "Janela de 730 dias nunca fica completa (dados com 2 anos) e é quase redundante com a de 180 dias (ρ ≥ 0,97).")
add(["post_no_previous_completed_session", "post_no_previous_completed_abnormal"],
    4, "Só valem 1 nas primeiras semanas (arranque, até mar/2020); em operação são constantes.")

# --- Categoria 1: tudo o resto é elegível (grupo para a ablação posterior)
atribuidas = {r[0] for r in regras}
GRUPOS = {
    "meteorologia": ["Temperature(℃)", "Relative Humidity(%)", "Precipitation(mm)"],
    "tarifa": ["electricity_price"],
    "localização": ["Charging Post ID", "Location Information", "District Name"],
}
def grupo(nome):
    for g, cols in GRUPOS.items():
        if nome in cols: return g
    if nome == "Start Time": return "tempo"
    if nome.startswith("post_"): return "histórico do posto"
    if nome.startswith("user_"): return "histórico do utilizador"
    return "?"

for c in COLUNAS_ORIGINAIS:
    if c not in atribuidas:
        just = ("Instante da previsão; não entra como tal, é a fonte de `hour` e `dow`." if c == "Start Time"
                else "Conhecida no início da sessão (ou histórico de sessões já concluídas).")
        add(c, 1, just)

# --- Variáveis derivadas (não existem no ficheiro original)
add(["hour", "dow"], 1, "Derivadas de Start Time (determinísticas, sem aprender dos dados).", derivada=True)
add(["post_last_completed_abnormal", "user_last_completed_abnormal"], 1,
    "Indicam se a última sessão concluída do posto/utilizador foi anómala (derivadas dos atributos históricos).", derivada=True)
add("is_weekend", 4, "Derivada de `dow` (redundante).", derivada=True)
add("month", 4, "Não usada: só 2 anos de dados, sem ciclo anual repetido suficiente para aprender sazonalidade.", derivada=True)

tabela = pd.DataFrame(regras, columns=["variável", "cat", "justificação", "derivada"])
tabela["categoria"] = tabela["cat"].map(CAT)
tabela["grupo"] = [grupo(v) if (c == 1 or v in ("hour", "dow")) else "" for v, c in zip(tabela["variável"], tabela["cat"])]
tabela.loc[tabela["variável"].isin(["hour", "dow"]), "grupo"] = "tempo"
tabela.loc[tabela["variável"].str.endswith("last_completed_abnormal"), "grupo"] = tabela["variável"].map(
    lambda v: "histórico do posto" if v.startswith("post_") else "histórico do utilizador")

# --- Verificações
originais = tabela[~tabela["derivada"]]["variável"]
assert set(originais) == set(COLUNAS_ORIGINAIS), set(COLUNAS_ORIGINAIS) ^ set(originais)
assert originais.is_unique
preditores = tabela[(tabela["cat"] == 1) & (tabela["variável"] != "Start Time")]
print(f"Colunas originais classificadas: {len(originais)}/{len(COLUNAS_ORIGINAIS)}")
print(f"Preditores elegíveis (sem Start Time): {len(preditores)}")
print(tabela.groupby("categoria")["variável"].count().to_string())

Colunas originais classificadas: 47/47
Preditores elegíveis (sem Start Time): 33
categoria
1. Preditor elegível                        34
2. Só para construir o alvo                  1
3. Gerada durante/depois da sessão           7
4. Excluída por outra razão metodológica    11


In [23]:
# Tabela completa por categoria (para o relatório: copiar as exclusões 2, 3 e 4)
tabela[tabela["cat"] != 1][["categoria", "variável", "justificação"]].sort_values(["categoria", "variável"]).reset_index(drop=True)

,categoria,variável,justificação
0,2. Só para construir o alvo,end_cause,Define o alvo e só é conhecida no fim da sessão (fuga de informação direta).
1,3. Gerada durante/depois da sessão,Actual Payment/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
2,3. Gerada durante/depois da sessão,Electricity cost/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
3,3. Gerada durante/depois da sessão,End Time,Só existe quando a sessão termina.
4,3. Gerada durante/depois da sessão,Payment time,Registado depois do fim da sessão.
5,3. Gerada durante/depois da sessão,Service charge/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
6,3. Gerada durante/depois da sessão,Transaction Amount/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
7,3. Gerada durante/depois da sessão,Transaction power/kwh,"Energia acumulada durante a sessão; as sessões anómalas têm energia nula ou duração curta (consequência do fim, não causa)."
8,4. Excluída por outra razão metodológica,Order creation time,"Na maioria das sessões coincide com o início, mas em parte das sessões é registada perto do fim (fuga de informação) e o sentido da diferença está associado ao alvo; não é fiável como informação de início."
9,4. Excluída por outra razão metodológica,UserID,Identificador de cardinalidade muito alta; grande parte das sessões de validação é de contas não vistas no treino (não generaliza) e identifica pessoas (privacidade). O comportamento do utilizador entra pelos atributos user_*.


In [24]:
# Preditores elegíveis por grupo (serão os grupos da ablação)
(preditores.groupby("grupo")["variável"].apply(list)
           .rename("variáveis").to_frame())

,variáveis
grupo,
histórico do posto,"[post_previous_sessions_1D, post_previous_abnormal_1D, post_previous_abnormal_rate_1D, post_previous_sessions_7D, post_previous_abnormal_7D, post_previous_abnormal_rate_7D, post_previous_sessions_30D, post_previous_abnormal_30D, post_previous_abnormal_rate_30D, post_hours_since_previous_completed_session, post_hours_since_previous_completed_abnormal, post_last_completed_abnormal]"
histórico do utilizador,"[user_previous_sessions_30D, user_previous_abnormal_30D, user_previous_abnormal_rate_30D, user_previous_sessions_180D, user_previous_abnormal_180D, user_previous_abnormal_rate_180D, user_days_since_previous_completed_session, user_no_previous_completed_session, user_days_since_previous_completed_abnormal, user_no_previous_completed_abnormal, user_active_sessions_at_start, user_last_completed_abnormal]"
localização,"[Charging Post ID, Location Information, District Name]"
meteorologia,"[Temperature(℃), Relative Humidity(%), Precipitation(mm)]"
tarifa,[electricity_price]
tempo,"[hour, dow]"


In [25]:
# ── Evidência 1: UserID — contas da validação que não existem no treino
vistos = set(train["UserID"])
nao_vistos = ~val["UserID"].isin(vistos)
print(f"Sessões de validação de contas não vistas no treino: {nao_vistos.mean():.1%}")

# ── Evidência 2: Order creation time — antes ou depois do início da sessão?
atraso_s = (dev["Order creation time"] - dev["Start Time"]).dt.total_seconds()      # >0: ordem criada DEPOIS do início
dist_fim_min = (dev["Order creation time"] - dev["End Time"]).dt.total_seconds().abs() / 60
tarde = atraso_s > 60 * 60
print(f"Mediana de (criação − início): {atraso_s.median():.0f} s")
print(f"Sessões com ordem criada mais de 60 min depois do início: {tarde.sum()} "
      f"({tarde.mean():.3%} do desenvolvimento)")
print(f"  dessas, registadas a ≤ 2 min do fim da sessão: {(dist_fim_min[tarde] <= 2).mean():.1%}")

Sessões de validação de contas não vistas no treino: 39.8%
Mediana de (criação − início): 8 s
Sessões com ordem criada mais de 60 min depois do início: 131 (0.036% do desenvolvimento)
  dessas, registadas a ≤ 2 min do fim da sessão: 73.3%


## 1.3 Falsos positivos vs falsos negativos

**O que fazer**

- [ ] Descrever as consequências de cada tipo de erro
- [ ] Listar as ações possíveis depois de um alerta e o custo de cada uma
- [ ] Decidir qual dos erros pesa mais e porquê
- [ ] Escrever o **critério do limiar** (isto é o que a Parte 2 precisa de receber)

**Rascunho para o relatório**

| Erro | O que acontece | Consequência |
|---|---|---|
| **Falso negativo** (sessão anómala não detetada) | Nenhum alerta; a falha ocorre sem aviso. | Sessão interrompida e utilizador prejudicado; a avaria tende a repetir-se, pois a Task 1 mostra que, quando a sessão anterior do posto foi anómala, a probabilidade de a seguinte também o ser sobe para cerca de 46 % (contra ~18 % de base); equipamento a falhar durante mais tempo sem intervenção. |
| **Falso positivo** (sessão normal sinalizada) | Alerta sem falha. | Inspeção ou intervenção desnecessária (custo de deslocação e de tempo), e **fadiga de alertas**: se muitos alertas forem falsos, a equipa deixa de os tratar. |

| Ação após o alerta | Custo | Tolera mais falsos positivos? |
|---|---|---|
| Monitorização automática reforçada do posto | Muito baixo | Sim |
| Priorização na fila de manutenção | Baixo a médio | Moderadamente |
| Inspeção técnica no local | Alto | Não |
| Intervenção junto do utilizador (aviso, sugestão de outro posto) | Médio (afeta a experiência) | Pouco |

**[DECISÃO]** O **falso negativo pesa mais**, porque as falhas são recorrentes e concentradas em postos que já falharam, pelo que deixar passar uma falha prolonga o problema. Mas o valor da deteção só se concretiza se a equipa conseguir tratar os alertas, o que impõe um **orçamento de alertas**. Daí o critério:

> **Maximizar o recall sujeito a um máximo de X alertas por 1000 sessões.**

Expressar o limiar como orçamento de alertas, e não como uma probabilidade fixa, também o torna mais robusto à variação da prevalência entre períodos (18,2 % no treino e 21,1 % na validação) e à recalibração do modelo quando for retreinado.

A célula seguinte mostra o que X significa em termos de recall alcançável, no treino e na validação.

In [26]:
# O que significa um orçamento de X alertas por 1000 sessões? (prevalência do treino)
prev = train["is_Abnormal"].mean()
prev_val = val["is_Abnormal"].mean()
X = np.array([50, 100, 150, 200, 300, 500])

meaning = pd.DataFrame({
    "alertas por 1000": X,
    "recall se o ranking fosse aleatório": X / 1000,
    "recall máx. possível (treino)": np.minimum(1, X / (prev * 1000)),
    "recall máx. possível (validação)": np.minimum(1, X / (prev_val * 1000)),
    "precisão máx. possível (treino)": np.minimum(1, prev * 1000 / X),
}).round(3)
print(f"Prevalência no treino: {prev:.1%} → ~{prev*1000:.0f} anómalas por 1000 sessões")
print(f"Prevalência na validação: {prev_val:.1%} → ~{prev_val*1000:.0f} anómalas por 1000 sessões")
meaning

Prevalência no treino: 18.2% → ~182 anómalas por 1000 sessões
Prevalência na validação: 21.1% → ~211 anómalas por 1000 sessões


,alertas por 1000,recall se o ranking fosse aleatório,recall máx. possível (treino),recall máx. possível (validação),precisão máx. possível (treino)
0,50,0.05,0.275,0.237,1.000
1,100,0.10,0.550,0.473,1.000
2,150,0.15,0.825,0.710,1.000
3,200,0.20,1.000,0.946,0.909
4,300,0.30,1.000,1.000,0.606
5,500,0.50,1.000,1.000,0.363


In [27]:
# >>> [DECISÃO] orçamento de alertas, fixado antes de treinar qualquer modelo <<<
X_ALERTAS_1000 = 200

**[DECISÃO] X = 200 alertas por 1000 sessões (20 % das sessões).**

1. Em cada 1000 sessões há cerca de 182 anómalas no treino e 211 na validação. X = 200 é da mesma ordem de grandeza do número de falhas esperadas; com X = 150, mesmo um modelo perfeito só detetaria cerca de 71 % das falhas na validação, o que seria demasiado restritivo.
2. A maioria das ações após um alerta é de custo baixo (monitorização reforçada, prioridade na fila de manutenção), o que tolera uma proporção elevada de falsos alertas.
3. As inspeções no local, que são caras, podem ser reservadas aos alertas com score mais alto dentro deste orçamento.

X foi fixado **antes de qualquer modelo ser treinado** e não será ajustado com base nos resultados. A Parte 2 reportará também a sensibilidade a X ∈ {100, 150, 200, 300}.

## 1.4 Métricas

**O que fazer**

- [ ] Para cada métrica: o que mede, porque é relevante e que limitação tem
- [ ] Justificar a métrica principal e porque a exatidão não serve
- [ ] Incluir métricas operacionais por 1000 sessões e de estabilidade temporal
- [ ] Lembrar que as métricas têm de estar justificadas **antes** da otimização

**Rascunho para o relatório**

| Métrica | Papel | O que mede | Porque é relevante | Limitação neste problema |
|---|---|---|---|---|
| **PR-AUC** (precisão média) | **Principal** | Compromisso entre precisão e recall ao longo de todos os limiares. | Centra-se na classe rara (anómala) e não depende da escolha do limiar, adequada para comparar e otimizar modelos. | Depende da prevalência, que varia entre períodos (18,2 % no treino e 21,1 % na validação); não é comparável entre folds sem normalização. |
| **PR-AUC ÷ prevalência** | Complementar | Quantas vezes o modelo supera um classificador aleatório (que tem PR-AUC ≈ prevalência). | Normaliza o efeito da prevalência entre folds. | Sozinha não diz se o desempenho é operacionalmente útil. |
| **ROC-AUC** | Complementar | Probabilidade de uma sessão anómala ter score superior ao de uma normal. | Insensível à prevalência; permite comparar com outros trabalhos. | Otimista com classes desbalanceadas, porque muitos negativos tornam fácil uma taxa de falsos positivos baixa. |
| **Recall** | Decisão | Fração das anómalas que geram alerta. | Traduz o custo do falso negativo, o erro que mais pesa. | Trivial (100 %) se alertarmos tudo; só tem sentido sob orçamento de alertas. |
| **Precisão** | Decisão | Fração dos alertas que são de facto anómalas. | Traduz o custo dos falsos positivos e a fadiga de alertas. | Depende do limiar e da prevalência. |
| **F2** | Decisão | Média harmónica de precisão e recall, com o recall a pesar 2×. | Resume o compromisso favorecendo o recall, coerente com 1.3. | Depende do limiar; um só número esconde o orçamento de alertas. |
| **Alertas corretos / falsos alertas / falhadas / total por 1000 sessões** | Operacional | Carga de trabalho e falhas deixadas passar, por 1000 sessões. | Dá uma leitura prática: quantos alertas a equipa recebe e quantas falhas escapam. | Dependem do limiar e da prevalência do período; só são comparáveis com o mesmo orçamento. |
| **Desvio-padrão entre folds, pior fold, métrica por mês** | Estabilidade | Variação do desempenho ao longo do tempo. | Um modelo instável não deve ganhar só por ter melhor média (enunciado). | Poucos folds (4) tornam o desvio-padrão pouco preciso; meses com poucas sessões geram valores ruidosos. |

**Exatidão (accuracy): não é usada.** Com 79 % a 82 % de sessões normais (conforme o período), um classificador que diga sempre "normal" já tem essa exatidão sem detetar nenhuma falha.

In [28]:
# Porque a exatidão não serve: classificador "sempre normal"
for nome, d in [("treino", train), ("validação", val)]:
    exat = 1 - d["is_Abnormal"].mean()
    print(f"{nome:10s} exatidão de 'sempre normal' = {exat:.1%} | anómalas detetadas = 0")

treino     exatidão de 'sempre normal' = 81.8% | anómalas detetadas = 0
validação  exatidão de 'sempre normal' = 78.9% | anómalas detetadas = 0


## 1.5 Função `avaliar()`

**O que fazer**

- [ ] Garantir que `avaliar(y, score, limiar)` devolve todas as métricas de 1.4
- [ ] Garantir que `limiar_por_orcamento()` traduz o critério de 1.3 num limiar
- [ ] Testar com casos em que se conhece a resposta

O código está em `avaliacao.py` (na raiz do repositório), para a Parte 2 o importar sem copiar. A célula seguinte mostra o código no notebook.

Convenção: `score` é maior quando o risco é maior; há alerta se `score >= limiar`. A estabilidade por mês usa `avaliar_por_mes()`.

In [29]:
print(inspect.getsource(avaliacao))

"""Avaliação dos modelos — Task 2

Funções partilhadas pela Parte 1 (métricas) e pela Parte 2 (folds e baselines).
Convenção: `score` = quanto maior, maior o risco de sessão anómala (classe 1).
"""
import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score, fbeta_score)


def limiar_por_orcamento(score, alertas_por_1000):
    """Limiar de score que gera ~`alertas_por_1000` alertas por 1000 sessões."""
    q = 1 - alertas_por_1000 / 1000
    return float(np.quantile(np.asarray(score, dtype=float), q))


def avaliar(y, score, limiar):
    """Métricas de ranking (independentes do limiar) e de decisão (com limiar).

    Alerta = score >= limiar.
    """
    y = np.asarray(y).astype(int)
    score = np.asarray(score, dtype=float)
    pred = (score >= limiar).astype(int)
    n = len(y)
    prev = y.mean()

    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred

In [30]:
# Testes com respostas conhecidas
rng = np.random.default_rng(0)
y_t = (rng.random(20_000) < 0.2).astype(int)

# 1) Score perfeito
r = avaliar(y_t, y_t.astype(float), 0.5)
assert r["pr_auc"] == 1 and r["roc_auc"] == 1 and r["recall"] == 1 and r["precisao"] == 1

# 2) Score constante (equivale a "sempre normal"): PR-AUC = prevalência, ROC-AUC = 0,5, nenhum alerta
r = avaliar(y_t, np.zeros(len(y_t)), 1.0)
assert np.isclose(r["pr_auc"], y_t.mean()) and np.isclose(r["roc_auc"], 0.5)
assert r["recall"] == 0 and r["total_alertas_1000"] == 0

# 3) Alertar tudo: recall 1, 1000 alertas por 1000 sessões, precisão = prevalência
r = avaliar(y_t, rng.random(len(y_t)), 0.0)
assert r["recall"] == 1 and r["total_alertas_1000"] == 1000 and np.isclose(r["precisao"], y_t.mean())

# 4) Orçamento: o limiar gera o nº de alertas pedido
s = rng.random(len(y_t))
r = avaliar(y_t, s, limiar_por_orcamento(s, X_ALERTAS_1000))
assert abs(r["total_alertas_1000"] - X_ALERTAS_1000) < 1

# 5) Coerência: alertas corretos + falhadas = anómalas por 1000 sessões
assert np.isclose(r["alertas_corretos_1000"] + r["falhadas_1000"], 1000 * y_t.mean())

# 6) Score aleatório: PR-AUC ≈ prevalência, ROC-AUC ≈ 0,5
r = avaliar(y_t, rng.random(len(y_t)), limiar_por_orcamento(s, X_ALERTAS_1000))
assert abs(r["pr_auc"] - y_t.mean()) < 0.02 and abs(r["roc_auc"] - 0.5) < 0.02

print("Todos os testes passaram.")

Todos os testes passaram.


In [31]:
# Demonstração das métricas por mês (dados sintéticos: serve para a Parte 2 ver o formato)
datas = pd.Series(pd.date_range("2021-01-01", periods=len(y_t), freq="h"))
score_demo = y_t * 0.5 + rng.random(len(y_t))
lim = limiar_por_orcamento(score_demo, X_ALERTAS_1000)
por_mes = avaliar_por_mes(datas, y_t, score_demo, lim)
print("ATENÇÃO: dados sintéticos, só para mostrar o formato. Não usar no relatório.")
por_mes[["n", "prevalencia", "pr_auc", "roc_auc", "recall", "total_alertas_1000"]].astype(float).round(3).head()

ATENÇÃO: dados sintéticos, só para mostrar o formato. Não usar no relatório.


,n,prevalencia,pr_auc,roc_auc,recall,total_alertas_1000
2021-01,744.0,0.198,0.729,0.872,0.599,211.022
2021-02,672.0,0.205,0.729,0.861,0.580,200.893
2021-03,744.0,0.237,0.723,0.851,0.540,193.548
2021-04,720.0,0.194,0.756,0.884,0.600,197.222
2021-05,744.0,0.199,0.786,0.893,0.689,217.742
